# Data Ingestion and SQL

## Getting Data Into Pandas

So far we've mostly used generated data or pre-loaded DataFrames. In practice, you'll need to load data from various sources:

- **CSV files** - The most common format for tabular data
- **JSON files** - Common for API responses and nested data
- **SQL databases** - Where most production data lives
- **APIs** - Real-time data from web services

In this notebook, we'll cover each of these, with a focus on SQL since it's the most important for professional work.

**Optional reference.** See the [lesson guide](../readme.md) for related notebooks.
 Run from a fresh kernel. The setup below locates the lesson folder from the repository root or either notebook folder.

In [ ]:
from pathlib import Path
import os
course_root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'pyproject.toml').is_file())
lesson_dir = course_root / 'Lectures' / 'Lecture 4'
os.chdir(lesson_dir)
(lesson_dir / 'outputs').mkdir(exist_ok=True)

In [ ]:
import pandas as pd
import numpy as np
import json
import sqlite3

print("Ready to load some data!")

---

# Part 1: CSV Files

CSV (Comma-Separated Values) is the simplest format for tabular data. It's just a text file where each line is a row and values are separated by commas.

In [ ]:
# Basic CSV reading
df = pd.read_csv('data/market_caps.csv')
df.head()

In [ ]:
# Pandas automatically infers types
df.dtypes

### Common `read_csv` Options

The `read_csv` function has many useful parameters:

In [ ]:
# Custom column names and index
df_custom = pd.read_csv(
    'data/market_caps.csv',
    header=0,                    # Row to use as column names (0 = first row)
    names=['Rank', 'Company', 'Symbol', 'MarketCap', 'Price'],  # Override column names
    index_col='Rank',            # Use this column as index
    skiprows=0,                  # Skip N rows at the start
    nrows=10                     # Only read first N rows
)
df_custom

In [ ]:
# Explicitly specify data types
df_typed = pd.read_csv(
    'data/market_caps.csv',
    dtype={
        'rank': 'int32',
        'name': 'string',
        'ticker': 'string',
        'market_cap': 'int64',
        'price': 'float64'
    }
)
df_typed.dtypes

### Quick Reference: `read_csv` Parameters

| Parameter | Purpose | Example |
|-----------|---------|--------|
| `sep` | Column separator | `sep='\t'` for tab-separated |
| `header` | Row number for column names | `header=None` if no header |
| `names` | Custom column names | `names=['A', 'B', 'C']` |
| `index_col` | Column to use as index | `index_col=0` or `index_col='date'` |
| `usecols` | Only read certain columns | `usecols=['date', 'close']` |
| `dtype` | Specify column types | `dtype={'id': 'int32'}` |
| `parse_dates` | Parse columns as dates | `parse_dates=['date']` |
| `skiprows` | Skip rows at start | `skiprows=5` |
| `nrows` | Only read N rows | `nrows=1000` |

---

# Part 2: JSON Files

JSON (JavaScript Object Notation) is common for API responses. It can represent nested structures that CSVs cannot.

In [ ]:
# If JSON is table-like, read_json works directly
df_json = pd.read_json('data/market_caps.json')
df_json.head()

### Handling Nested JSON

Often JSON from APIs has nested structure. Let's look at a more complex example:

In [ ]:
# Load JSON manually to see its structure
with open('data/payload.json', 'r') as f:
    data = json.load(f)

# Show the top-level keys
print("Top-level keys:")
print(list(data.keys()))

In [ ]:
# Look at the metadata
print("\nMetadata:")
for key, value in data['Meta Data'].items():
    print(f"  {key}: {value}")

In [ ]:
# The actual data is nested under 'Time Series (Daily)'
time_series = data['Time Series (Daily)']

# Show first entry
first_date = list(time_series.keys())[0]
print(f"First date: {first_date}")
print(f"Data: {time_series[first_date]}")

In [ ]:
# Convert nested JSON to DataFrame
df_ts = pd.DataFrame.from_dict(time_series, orient='index')

# # Clean up column names
df_ts.columns = ['open', 'high', 'low', 'close', 'volume']

# # Convert index to datetime
df_ts.index = pd.to_datetime(df_ts.index)
df_ts.index.name = 'date'

# Convert strings to numbers
for col in df_ts.columns:
    df_ts[col] = pd.to_numeric(df_ts[col])

# # Sort by date
df_ts = df_ts.sort_index()

df_ts.head(10)

---

# Part 3: SQL Databases

Most production data lives in SQL databases. SQL (Structured Query Language) is a language for querying relational databases.

We'll use SQLite, a file-based database that's great for learning and small projects. The concepts transfer directly to larger databases like PostgreSQL, MySQL, or SQL Server.

## SQL Concepts

**Connection**: A link to the database. You need this to send queries.

**Cursor**: An object that executes queries and fetches results.

**Query**: A SQL statement that retrieves or modifies data.

In [ ]:
# Connect to SQLite database
database = lesson_dir / 'data/data.db'
if not database.is_file():
    raise FileNotFoundError(f'Download data.db from bCourses to {database}')
conn = sqlite3.connect(database.resolve().as_uri() + '?mode=ro', uri=True)

# Create a cursor
cursor = conn.cursor()

print("Connected to database")

In [ ]:
# List all tables in the database
cursor.execute("SELECT name FROM sqlite_master WHERE type='table'")
tables = cursor.fetchall()

print("Tables in database:")
for table in tables:
    print(f"  - {table[0]}")

In [ ]:
# View schema for a table
cursor.execute("SELECT sql FROM sqlite_master WHERE name='ohlc'")
schema = cursor.fetchone()[0]
print("Schema for 'ohlc' table:")
print(schema)

## Basic SQL Queries

### SELECT: Retrieve Data

The basic syntax is:
```sql
SELECT columns FROM table WHERE conditions
```

In [ ]:
# Select all columns, limit rows
cursor.execute("SELECT * FROM ohlc LIMIT 5")
rows = cursor.fetchall()

print("First 5 rows:")
for row in rows:
    print(row)

In [ ]:
# Select specific columns with a WHERE clause
cursor.execute("""
    SELECT ts, ticker, close, volume 
    FROM ohlc 
    WHERE ticker = 'AAPL' 
    ORDER BY ts DESC
    LIMIT 10
""")

rows = cursor.fetchall()
print("Recent AAPL prices:")
for row in rows:
    print(f"  {row[0]}: ${row[2]:.2f} (volume: {row[3]:,})")

### WHERE: Filtering Data

Common operators:
- `=`, `!=`, `<`, `>`, `<=`, `>=` - Comparisons
- `AND`, `OR`, `NOT` - Logical operators
- `IN (...)` - Match any value in list
- `BETWEEN x AND y` - Range
- `LIKE` - Pattern matching

In [ ]:
# Multiple conditions
cursor.execute("""
    SELECT ts, ticker, close 
    FROM ohlc 
    WHERE ticker IN ('AAPL', 'MSFT', 'GOOGL')
      AND ts >= '2024-01-01'
      AND close > 150
    ORDER BY ticker, ts
    LIMIT 15
""")

for row in cursor.fetchall():
    print(f"{row[0]} {row[1]}: ${row[2]:.2f}")

### Aggregation: GROUP BY

Aggregate functions:
- `COUNT(*)` - Number of rows
- `SUM(column)` - Total
- `AVG(column)` - Average
- `MIN(column)`, `MAX(column)` - Extremes

In [ ]:
# Summary statistics by ticker
cursor.execute("""
    SELECT 
        ticker,
        COUNT(*) as num_days,
        MIN(ts) as first_date,
        MAX(ts) as last_date,
        AVG(close) as avg_close,
        MIN(close) as min_close,
        MAX(close) as max_close
    FROM ohlc
    WHERE ticker IN ('AAPL', 'MSFT', 'BA')
    GROUP BY ticker
    ORDER BY avg_close DESC
""")

print(f"{'Ticker':<8} {'Days':>6} {'Avg Close':>12} {'Min':>10} {'Max':>10}")
print("-" * 50)
for row in cursor.fetchall():
    print(f"{row[0]:<8} {row[1]:>6} ${row[4]:>10.2f} ${row[5]:>9.2f} ${row[6]:>9.2f}")

### JOIN: Combining Tables

JOINs let you combine data from multiple tables based on matching columns.

In [ ]:
# Join OHLC data with market cap data
cursor.execute("""
    SELECT 
        o.ts,
        o.ticker,
        o.close,
        m.market_capitalization / 1e9 as market_cap_billions
    FROM ohlc o
    JOIN market_caps m 
        ON o.ticker = m.ticker 
        AND o.ts = m.ts
    WHERE o.ticker = 'AAPL'
    ORDER BY o.ts DESC
    LIMIT 10
""")

print(f"{'Date':<12} {'Ticker':<6} {'Close':>10} {'Market Cap ($B)':>15}")
print("-" * 50)
for row in cursor.fetchall():
    print(f"{row[0]:<12} {row[1]:<6} ${row[2]:>9.2f} ${row[3]:>14.1f}")

---

# Part 4: SQL + Pandas Integration

While you can use cursors directly, pandas provides a much easier way: `pd.read_sql()`

In [ ]:
# Read SQL query directly into DataFrame
df = pd.read_sql("""
    SELECT * FROM ohlc 
    WHERE ticker = 'AAPL' 
    AND ts >= '2024-01-01'
    ORDER BY ts
""", conn)

print(f"Loaded {len(df)} rows")
df.head()

In [ ]:
# Pandas handles type conversion
df.dtypes

In [ ]:
# Parse dates and set index
df = pd.read_sql("""
    SELECT * FROM ohlc 
    WHERE ticker = 'AAPL' 
    AND ts >= '2024-01-01'
    ORDER BY ts
""", conn, parse_dates=['ts'], index_col='ts')

df.head()

### Complex Queries

You can use any valid SQL, including CTEs (Common Table Expressions):

In [ ]:
# Calculate average daily volume by ticker for 2024
df_volume = pd.read_sql("""
    WITH daily_volume AS (
        SELECT 
            ticker,
            ts,
            volume,
            adj_close * volume as dollar_volume
        FROM ohlc
        WHERE ts >= '2024-01-01'
    )
    SELECT 
        ticker,
        COUNT(*) as trading_days,
        AVG(volume) as avg_volume,
        AVG(dollar_volume) / 1e6 as avg_dollar_volume_millions
    FROM daily_volume
    GROUP BY ticker
    ORDER BY avg_dollar_volume_millions DESC
    LIMIT 10
""", conn)

df_volume

### Writing DataFrames to SQL

You can also write DataFrames back to the database:

In [ ]:
# Create a summary DataFrame
summary = pd.read_sql("""
    SELECT 
        ticker,
        AVG(close) as avg_close,
        AVG(volume) as avg_volume
    FROM ohlc
    WHERE ts >= '2024-01-01'
    GROUP BY ticker
""", conn)

# Write to new table
demo_conn = sqlite3.connect(':memory:')
summary.to_sql('ticker_summary_2024', demo_conn, if_exists='replace', index=False)

print("Created table 'ticker_summary_2024'")

# Verify it was created
pd.read_sql("SELECT * FROM ticker_summary_2024 LIMIT 5", demo_conn)

In [ ]:
demo_conn.execute('DROP TABLE IF EXISTS ticker_summary_2024')
demo_conn.close()
print('Closed the in-memory demonstration database; source data was read-only.')

### SQL vs Pandas: When to Use Which?

**Do filtering/aggregation in SQL when:**
- Data is very large (millions of rows)
- You only need a subset of the data
- The database can do the computation faster

**Do it in Pandas when:**
- You need complex transformations SQL can't do easily
- You're working iteratively and need to see results
- Data is small enough to fit in memory

**Typical workflow:**
1. Use SQL to filter and do initial aggregation
2. Load the reduced dataset into pandas
3. Do detailed analysis in pandas

In [ ]:
# Example workflow: Analyze AAPL volatility

# Step 1: Use SQL to get just the data we need
df_aapl = pd.read_sql("""
    SELECT ts, open, high, low, close, adj_close, volume
    FROM ohlc
    WHERE ticker = 'AAPL'
      AND ts >= '2020-01-01'
    ORDER BY ts
""", conn, parse_dates=['ts'], index_col='ts')

print(f"Loaded {len(df_aapl)} rows of AAPL data")

# Step 2: Do analysis in pandas
df_aapl['returns'] = df_aapl['adj_close'].pct_change(fill_method=None)
df_aapl['volatility_20d'] = df_aapl['returns'].rolling(20).std() * np.sqrt(252)

# Step 3: Summarize
print(f"\nAAPL Statistics (2020-present):")
print(f"  Average daily return: {df_aapl['returns'].mean()*100:.3f}%")
print(f"  Average 20-day volatility: {df_aapl['volatility_20d'].mean()*100:.1f}%")
print(f"  Current 20-day volatility: {df_aapl['volatility_20d'].iloc[-1]*100:.1f}%")

---

# Part 5: Practice with the Homework Data

The database has a table called `ohlc_hw` that you'll use for homework. Let's explore it:

In [ ]:
# Check the schema
cursor.execute("SELECT sql FROM sqlite_master WHERE name='ohlc_hw'")
print("Schema:")
print(cursor.fetchone()[0])

In [ ]:
# Quick summary of the homework data
df_hw_summary = pd.read_sql("""
    SELECT 
        ticker,
        COUNT(*) as num_rows,
        MIN(ts) as first_date,
        MAX(ts) as last_date
    FROM ohlc_hw
    GROUP BY ticker
    ORDER BY ticker
""", conn)

print(f"Homework data contains {len(df_hw_summary)} tickers:")
df_hw_summary

In [ ]:
# Look at sample data
df_hw_sample = pd.read_sql("""
    SELECT * FROM ohlc_hw 
    WHERE ticker = 'AAPL'
    ORDER BY ts DESC
    LIMIT 20
""", conn)

print("Sample from ohlc_hw (AAPL):")
df_hw_sample

### Hints for the Homework

The `ohlc_hw` table has the same structure as `ohlc`, but may have data quality issues that you'll need to identify and clean.

**Useful queries to explore:**

```sql
-- Check for missing values
SELECT COUNT(*) as total, 
       COUNT(close) as non_null_close
FROM ohlc_hw

-- Check for suspicious prices
SELECT * FROM ohlc_hw 
WHERE close < 0 OR close > 10000

-- Check for duplicates
SELECT ts, ticker, COUNT(*) 
FROM ohlc_hw 
GROUP BY ts, ticker 
HAVING COUNT(*) > 1
```

---

# Summary

## Data Ingestion Methods

| Source | Function | When to Use |
|--------|----------|-------------|
| CSV | `pd.read_csv()` | Simple tabular data |
| JSON | `pd.read_json()` or `json.load()` | API responses, nested data |
| SQL | `pd.read_sql()` | Production databases |

## SQL Quick Reference

```sql
-- Select with filtering
SELECT columns FROM table WHERE conditions ORDER BY column LIMIT n

-- Aggregation
SELECT ticker, AVG(close) FROM table GROUP BY ticker

-- Join tables
SELECT * FROM table1 JOIN table2 ON table1.key = table2.key
```

## Pandas + SQL

```python
# Read
conn = sqlite3.connect('database.db')
df = pd.read_sql("SELECT * FROM table", conn)

# Write
df.to_sql('table_name', conn, if_exists='replace')
```

In [ ]:
# Close database connection
conn.close()
print("Database connection closed")